# 02 数据清洗与日销量面板构造

本 Notebook 用于阶段 1 的预处理复现，生成日销量面板和建模基础表。

## 代码说明：读取并标准化原始数据

这段代码把中文原始字段统一为英文标准字段，并从 `收款时间` 提取日期。这样处理是为了让后续代码稳定复用。

In [ ]:
from src.data_loader import load_raw_sales, load_raw_weather, save_processed
from src.preprocessing import standardize_sales, standardize_weather, build_daily_store_product_panel, mark_daily_sales_outliers, merge_external_variables

raw_sales = load_raw_sales()
raw_weather = load_raw_weather()
sales = standardize_sales(raw_sales)
weather = standardize_weather(raw_weather)
display(sales.head())
display(weather.head())


输出怎么看：`sales` 应包含 `date`、`weekday`、`is_weekend`、`month`；`weather` 应包含 `is_holiday`、`weekday_from_date` 等字段。若日期解析失败，相关列会出现缺失。

## 代码说明：构造日期-门店-商品日销量面板

这段代码把交易明细按 `date + store + product` 汇总，并补齐无销售日期为 0。这样处理是因为预测模型需要每天一行，而不是一笔交易一行。

In [ ]:
daily_panel = build_daily_store_product_panel(sales, complete_dates=True)
daily_panel = mark_daily_sales_outliers(daily_panel)
print(daily_panel.shape)
display(daily_panel.head())
print('净日销量为负的行:', (daily_panel['daily_sales'] < 0).sum())
print('日销量异常标记行:', daily_panel['daily_sales_outlier'].sum())


输出怎么看：`daily_sales` 是日净销量，`positive_sales` 和 `negative_adjustment_qty` 用于区分正常销售与损耗/冲销类负向调整。异常标记只是提示风险，不代表删除。

## 代码说明：合并外部变量并保存

这段代码按 `date` 合并天气、节日、活动日等变量，并保存两个 CSV。这样处理是为了给问题三和问题四准备统一建模表。

In [ ]:
modeling_base = merge_external_variables(daily_panel, weather)
print(modeling_base.shape)
print('外部变量缺失行:', (modeling_base['has_external_data'] == 0).sum())
display(modeling_base.head())
save_processed(daily_panel, 'daily_store_product_sales.csv')
save_processed(modeling_base, 'modeling_base_table.csv')


输出怎么看：如果 `has_external_data=0`，说明该销售日期没有匹配到附件二。当前主要缺口来自销售最后一天和未来预测期天气不可得，后续建模要避免依赖未知未来天气。